In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# =========================================================
# 1. 数据读取与 ROI
# =========================================================
def load_events(file_path):
    with h5py.File(file_path, 'r') as f:
        e = f['CD/events'][:]
        return {
            'x': e['x'].astype(float),
            'y': e['y'].astype(float),
            't': e['t'].astype(float) * 1e-6
        }

def select_large_roi(data, x_min, x_max, y_min, y_max, t_min, t_max):
    mask = (data['x'] >= x_min) & (data['x'] <= x_max) & \
           (data['y'] >= y_min) & (data['y'] <= y_max) & \
           (data['t'] >= t_min) & (data['t'] <= t_max)
    return {k: v[mask] for k, v in data.items()}

# =========================================================
# 2. 空间分组
# =========================================================
def group_by_x(roi_data, group_size=200):
    x_min, x_max = roi_data['x'].min(), roi_data['x'].max()
    bounds = np.arange(x_min, x_max + group_size, group_size)
    groups = []
    for i in range(len(bounds) - 1):
        x0, x1 = bounds[i], bounds[i + 1]
        mask = (roi_data['x'] >= x0) & (roi_data['x'] < x1)
        if np.sum(mask) < 100: continue
        groups.append({'id': i, 'x_range': (x0, x1), 'data': {k: v[mask] for k, v in roi_data.items()}})
    return groups

# =========================================================
# 3. 隐式模型与拟合
# =========================================================
def fit_group_implicit(group):
    data = group['data']
    x_raw, y_raw, t_raw = data['x'], data['y'], data['t']
    
    # --- 数值稳定性：去中心化 ---
    x_m, y_m = np.mean(x_raw), np.mean(y_raw)
    t_0 = t_raw.min()
    x, y, t = x_raw - x_m, y_raw - y_m, t_raw - t_0

    # 初始频率估计 (利用 y 轴方向波动作初值)
    y_c = y - np.mean(y)
    freqs = np.linspace(1, 300, 200)
    power = [np.abs(np.sum(y_c * np.exp(-1j*2*np.pi*f*t))) for f in freqs]
    f_init = freqs[np.argmax(power)]

    # 参数：[A, B, C, f, phi, lamb, D]
    # 隐式方程：Ax + By + C*exp(-lamb*t)*sin(2*pi*f*t + phi) + D = 0
    p0 = [0.0, 1.0, np.ptp(y)/2, f_init, 0.0, 0.1, 0.0]

    def objective(p):
        A, B, C, f, phi, lamb, D = p
        # 几何约束：A^2 + B^2 = 1 保证残差即为法向距离
        constraint_penalty = 1e6 * (np.sqrt(A**2 + B**2) - 1)**2
        
        vibration = C * np.exp(-lamb * t) * np.sin(2 * np.pi * f * t + phi)
        residual = A * x + B * y + vibration + D
        return np.mean(residual**2) + constraint_penalty

    res = minimize(objective, p0, method='L-BFGS-B', 
                   bounds=[(-1,1), (-1,1), (0,None), (1,500), (-np.pi, np.pi), (0,10), (None,None)])
    
    if not res.success: return None
    
    # 归一化结果
    popt = res.x
    norm = np.sqrt(popt[0]**2 + popt[1]**2)
    popt[[0,1,2,6]] /= norm

    return {
        'params': popt, 'x_m': x_m, 'y_m': y_m, 't_0': t_0, 'id': group['id'], 'x_range': group['x_range']
    }

# =========================================================
# 4. 可视化
# =========================================================
def plot_raw_and_surface(roi_data, fitted_groups):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # --- 1. 先可视化原始散点数据 ---
    # 使用较小的 s (点大小) 和 alpha (透明度) 以免遮挡曲面
    ax.scatter(
        roi_data['x'], 
        roi_data['t'], 
        roi_data['y'], 
        s=1, 
        alpha=0.15, 
        c='dodgerblue', 
        label='Raw Events'
    )

    # --- 2. 再可视化拟合的曲面 ---
    for g in fitted_groups:
        p = g['params']
        A, B, C, f, phi, lamb, D = p
        x0, x1 = g['x_range']
        
        # 生成网格数据（用于绘制曲面）
        xg = np.linspace(x0, x1, 50)
        tg = np.linspace(roi_data['t'].min(), roi_data['t'].max(), 100)
        XG, TG = np.meshgrid(xg, tg)
        
        # 局部坐标转换
        xl = XG - g['x_m']
        tl = TG - g['t_0']
        
        # 计算隐式曲面的 Y 值 (基于去中心化参数)
        vibration = C * np.exp(-lamb * tl) * np.sin(2 * np.pi * f * tl + phi)
        # yl = -(Ax + V + D) / B
        yl = -(A * xl + vibration + D) / B
        
        # 还原回全局 Y 坐标
        YG = yl + g['y_m']
        
        # 绘制曲面，使用 alpha=0.6 保持半透明，以便观察内部点云
        surf = ax.plot_surface(
            XG, TG, YG, 
            cmap='viridis', 
            alpha=0.6, 
            linewidth=0, 
            antialiased=True
        )

    # 设置标签和视角
    ax.set_xlabel("X (Pixels)")
    ax.set_ylabel("Time (s)")
    ax.set_zlabel("Y (Pixels)")
    ax.set_title("3DFIT: Raw Events vs Fitted Damped Surface")
    
    # 调整视角以便更好地观察拟合贴合度
    ax.view_init(elev=20, azim=-35) 
    
    plt.tight_layout()
    plt.show()
def plot_raw_events_only(roi_data):
    """图 1：仅展示原始事件散点图"""
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(
        roi_data['x'], 
        roi_data['t'], 
        roi_data['y'], 
        s=1, 
        alpha=0.3, 
        c='dodgerblue', 
        edgecolors='none'
    )

    ax.set_xlabel("X (Pixels)")
    ax.set_ylabel("Time (s)")
    ax.set_zlabel("Y (Pixels)")
    ax.set_title("Figure 1: Raw Event Spatiotemporal Cloud")
    ax.view_init(elev=20, azim=-35)
    plt.tight_layout()
    plt.show()

def plot_fitted_surface_only(roi_data, fitted_groups):
    """图 2：仅展示拟合后的隐式阻尼曲面"""
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    for g in fitted_groups:
        p = g['params']
        A, B, C, f, phi, lamb, D = p
        x0, x1 = g['x_range']
        
        # 生成网格
        xg = np.linspace(x0, x1, 60)
        tg = np.linspace(roi_data['t'].min(), roi_data['t'].max(), 120)
        XG, TG = np.meshgrid(xg, tg)
        
        # 局部坐标计算
        xl = XG - g['x_m']
        tl = TG - g['t_0']
        
        # 隐式转显式计算 YG
        vibration = C * np.exp(-lamb * tl) * np.sin(2 * np.pi * f * tl + phi)
        yl = -(A * xl + vibration + D) / B
        YG = yl + g['y_m']
        
        # 绘制曲面
        surf = ax.plot_surface(
            XG, TG, YG, 
            cmap='viridis', 
            alpha=0.9, 
            antialiased=True,
            rcount=100, ccount=100
        )

    ax.set_xlabel("X (Pixels)")
    ax.set_ylabel("Time (s)")
    ax.set_zlabel("Y (Pixels)")
    ax.set_title("Figure 2: Fitted Implicit Damped Surface")
    ax.view_init(elev=20, azim=-35)
    plt.tight_layout()
    plt.show()
# =========================================================
# 5. Main
# =========================================================
def main():
    # file_path = "./data/zdbiaochi.hdf5" # 替换为你的路径
    # # data = load_events(file_path)
    # # roi = select_large_roi(data, 300, 1000, 400, 600, 0, 0.2)

    # data = load_events(file_path)

    # roi = select_large_roi(
    #     data,
    #     300, 1000,
    #     400, 600,
    #     0, 0.2
    # )
    
    
    # file_path = './data/200hz-20.hdf5'  # 替换为你的HDF5文件路径
    # data = load_events(file_path)

    # roi = select_large_roi(
    #     data,
    #     200, 600,
    #     350, 460,
    #     0, 0.5
    # )


    file_path = "./data/30hz-20.hdf5"
    data = load_events(file_path)
    roi = select_large_roi(
        data,
        100, 600,
        420, 460,
        0, 1.0
    )    
    
    
    print(f"Events in ROI: {len(roi['x'])}")
    groups = group_by_x(roi, group_size=800)
    
    fitted = []
    for g in groups:
        res = fit_group_implicit(g)
        if res:
            fitted.append(res)
            print(f"Group {res['id']} Fitted: f={res['params'][3]:.2f}Hz, λ={res['params'][5]:.2f}")

    # if fitted:
    #     plot_raw_and_surface(roi, fitted)


# 假设得到的拟合结果存储在 fitted 列表中
    if not fitted:
        print("拟合失败，无数据可绘。")
        return

    # --- 弹出图 1：原始数据 ---
    print("正在生成图 1：原始散点...")
    plot_raw_events_only(roi)

    # --- 弹出图 2：拟合曲面 ---
    print("正在生成图 2：拟合曲面...")
    plot_fitted_surface_only(roi, fitted)
if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# =========================================================
# 1. 数据读取与 ROI
# =========================================================
def load_events(file_path):

    with h5py.File(file_path, 'r') as f:

        e = f['CD/events'][:]

        return {
            'x': e['x'].astype(float),
            'y': e['y'].astype(float),
            't': e['t'].astype(float) * 1e-6
        }


def select_large_roi(
        data,
        x_min, x_max,
        y_min, y_max,
        t_min, t_max):

    mask = (
        (data['x'] >= x_min) &
        (data['x'] <= x_max) &
        (data['y'] >= y_min) &
        (data['y'] <= y_max) &
        (data['t'] >= t_min) &
        (data['t'] <= t_max)
    )

    return {
        k: v[mask]
        for k, v in data.items()
    }


# =========================================================
# 2. 空间分组
# =========================================================
def group_by_x(
        roi_data,
        group_size=200):

    x_min = roi_data['x'].min()
    x_max = roi_data['x'].max()

    bounds = np.arange(
        x_min,
        x_max + group_size,
        group_size
    )

    groups = []

    for i in range(len(bounds) - 1):

        x0 = bounds[i]
        x1 = bounds[i + 1]

        mask = (
            (roi_data['x'] >= x0) &
            (roi_data['x'] < x1)
        )

        if np.sum(mask) < 100:
            continue

        groups.append({
            'id': i,
            'x_range': (x0, x1),
            'data': {
                k: v[mask]
                for k, v in roi_data.items()
            }
        })

    return groups


# =========================================================
# 3. 3DFIT 拟合
# =========================================================
def fit_group_implicit(group):

    data = group['data']

    x_raw = data['x']
    y_raw = data['y']
    t_raw = data['t']

    # -----------------------------------------------------
    # 去中心化
    # -----------------------------------------------------
    x_m = np.mean(x_raw)
    y_m = np.mean(y_raw)
    t_0 = t_raw.min()

    x = x_raw - x_m
    y = y_raw - y_m
    t = t_raw - t_0

    # -----------------------------------------------------
    # 初始频率估计
    # -----------------------------------------------------
    y_c = y - np.mean(y)

    freqs = np.linspace(1, 300, 200)

    power = [
        np.abs(
            np.sum(
                y_c *
                np.exp(-1j * 2 * np.pi * f * t)
            )
        )
        for f in freqs
    ]

    f_init = freqs[np.argmax(power)]

    # -----------------------------------------------------
    # 初始参数
    # -----------------------------------------------------
    p0 = [
        0.0,
        1.0,
        np.ptp(y) / 2,
        f_init,
        0.0,
        0.1,
        0.0
    ]

    # -----------------------------------------------------
    # 目标函数
    # -----------------------------------------------------
    def objective(p):

        A, B, C, f, phi, lamb, D = p

        # 几何约束
        penalty = (
            1e6 *
            (np.sqrt(A**2 + B**2) - 1)**2
        )

        vibration = (
            C *
            np.exp(-lamb * t) *
            np.sin(
                2 * np.pi * f * t + phi
            )
        )

        residual = (
            A * x +
            B * y +
            vibration +
            D
        )

        return (
            np.mean(residual**2) +
            penalty
        )

    # -----------------------------------------------------
    # 优化
    # -----------------------------------------------------
    res = minimize(
        objective,
        p0,
        method='L-BFGS-B',
        bounds=[
            (-1, 1),
            (-1, 1),
            (0, None),
            (1, 500),
            (-np.pi, np.pi),
            (0, 10),
            (None, None)
        ]
    )

    if not res.success:
        return None

    popt = res.x

    # -----------------------------------------------------
    # 归一化
    # -----------------------------------------------------
    norm = np.sqrt(
        popt[0]**2 +
        popt[1]**2
    )

    popt[[0,1,2,6]] /= norm

    return {
        'params': popt,
        'x_m': x_m,
        'y_m': y_m,
        't_0': t_0,
        'id': group['id'],
        'x_range': group['x_range']
    }


# =========================================================
# 4. 使用拟合参数重建信号
# =========================================================
def reconstruct_signal(
        fit_result,
        roi_data,
        dt=0.001):

    # -----------------------------------------------------
    # 读取参数
    # -----------------------------------------------------
    p = fit_result['params']

    A, B, C, f, phi, lamb, D = p

    # -----------------------------------------------------
    # 时间轴
    # 与ECT/Gabor统一
    # -----------------------------------------------------
    t_min = roi_data['t'].min()
    t_max = roi_data['t'].max()

    times = np.arange(
        t_min,
        t_max,
        dt
    )

    tl = times - fit_result['t_0']

    # -----------------------------------------------------
    # 使用拟合参数直接生成振动信号
    # =====================================================
    # 关键：
    # 这里的频率就是拟合出的 f
    # 不再重新估计
    # -----------------------------------------------------
    signal = (
        C *
        np.exp(-lamb * tl) *
        np.sin(
            2 * np.pi * f * tl + phi
        )
    )

    signal -= np.mean(signal)

    return times, signal, f, C


# =========================================================
# 5. 频谱可视化（ECT风格）
# =========================================================
def plot_3dfit_spectrum(
        times,
        signal,
        fitted_freq):

    # -----------------------------------------------------
    # 可视化参数
    # -----------------------------------------------------
    grid_alpha = 0.3
    label_fontsize = 20
    tick_fontsize = 18

    # -----------------------------------------------------
    # FFT
    # -----------------------------------------------------
    fs = 1.0 / np.mean(np.diff(times))

    N = len(signal)

    freqs = np.fft.rfftfreq(
        N,
        d=1/fs
    )

    spectrum = np.abs(
        np.fft.rfft(signal)
    )

    # -----------------------------------------------------
    # 归一化
    # -----------------------------------------------------
    spectrum /= (
        spectrum.max() + 1e-12
    )

    # -----------------------------------------------------
    # 不再使用 FFT peak
    # 而是：
    # 使用拟合频率对应的位置
    # -----------------------------------------------------
    idx = np.argmin(
        np.abs(freqs - fitted_freq)
    )

    peak_freq = freqs[idx]
    peak_amp = spectrum[idx]

    # =====================================================
    # 频谱图
    # =====================================================
    plt.figure(figsize=(6, 5))

    plt.plot(
        freqs,
        spectrum,
        lw=2
    )

    # -----------------------------------------------------
    # 红点：拟合频率
    # -----------------------------------------------------
    plt.scatter(
        peak_freq,
        peak_amp,
        color='r',
        s=60,
        zorder=3,
        label=f"{fitted_freq:.2f} Hz"
    )

    plt.xlabel(
        "Frequency (Hz)",
        fontsize=label_fontsize
    )

    plt.ylabel(
        "Normalized Amplitude",
        fontsize=label_fontsize
    )

    plt.xlim(0, 250)

    plt.legend(
        frameon=False,
        fontsize=tick_fontsize,
        loc='upper right'
    )

    plt.grid(alpha=grid_alpha)

    plt.tick_params(
        axis='both',
        labelsize=tick_fontsize
    )

    plt.tight_layout()

    plt.show()


# =========================================================
# 6. main
# =========================================================
def main():

    # -----------------------------------------------------
    # 数据
    # -----------------------------------------------------
    file_path = "./data/30hz-20.hdf5"

    data = load_events(file_path)

    roi = select_large_roi(
        data,
        100, 600,
        420, 460,
        0, 1.0
    )

    print(
        f"Events in ROI: {len(roi['x'])}"
    )

    # -----------------------------------------------------
    # 分组
    # -----------------------------------------------------
    groups = group_by_x(
        roi,
        group_size=800
    )

    fitted = []

    # -----------------------------------------------------
    # 拟合
    # -----------------------------------------------------
    for g in groups:

        res = fit_group_implicit(g)

        if res:

            fitted.append(res)

            print(
                f"Group {res['id']} "
                f"Fitted: "
                f"f={res['params'][3]:.2f}Hz, "
                f"λ={res['params'][5]:.2f}"
            )

    if not fitted:

        print("拟合失败")
        return

    # =====================================================
    # 使用第一个group
    # =====================================================
    fit_result = fitted[0]

    # -----------------------------------------------------
    # 重建信号
    # -----------------------------------------------------
    times, signal, fitted_freq, amplitude = \
        reconstruct_signal(
            fit_result,
            roi,
            dt=0.001
        )

    # -----------------------------------------------------
    # 频谱图
    # -----------------------------------------------------
    plot_3dfit_spectrum(
        times,
        signal,
        fitted_freq
    )

    # =====================================================
    # 输出
    # =====================================================
    print("\n========== 3DFIT Result ==========")

    print(
        f"Fitted Frequency : "
        f"{fitted_freq:.3f} Hz"
    )

    print(
        f"Estimated Amplitude : "
        f"{amplitude:.3f} px"
    )


# =========================================================
# main
# =========================================================
if __name__ == "__main__":

    main()

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# =========================================================
# 1. 数据读取
# =========================================================
def load_events(file_path):

    with h5py.File(file_path, 'r') as f:

        e = f['CD/events'][:]

        return {
            'x': e['x'].astype(float),
            'y': e['y'].astype(float),
            't': e['t'].astype(float) * 1e-6
        }


# =========================================================
# 2. ROI 截取
# =========================================================
def select_large_roi(data,
                     x_min, x_max,
                     y_min, y_max,
                     t_min, t_max):

    mask = (
        (data['x'] >= x_min) &
        (data['x'] <= x_max) &
        (data['y'] >= y_min) &
        (data['y'] <= y_max) &
        (data['t'] >= t_min) &
        (data['t'] <= t_max)
    )

    return {
        k: v[mask]
        for k, v in data.items()
    }


# =========================================================
# 3. 空间分组
# =========================================================
def group_by_x(roi_data,
               group_size=200):

    x_min = roi_data['x'].min()
    x_max = roi_data['x'].max()

    bounds = np.arange(
        x_min,
        x_max + group_size,
        group_size
    )

    groups = []

    for i in range(len(bounds) - 1):

        x0 = bounds[i]
        x1 = bounds[i + 1]

        mask = (
            (roi_data['x'] >= x0) &
            (roi_data['x'] < x1)
        )

        if np.sum(mask) < 100:
            continue

        groups.append({
            'id': i,
            'x_range': (x0, x1),
            'data': {
                k: v[mask]
                for k, v in roi_data.items()
            }
        })

    return groups


# =========================================================
# 4. 3DFIT 拟合
# =========================================================
def fit_group_implicit(group):

    data = group['data']

    x_raw = data['x']
    y_raw = data['y']
    t_raw = data['t']

    # -----------------------------------------------------
    # 去中心化
    # -----------------------------------------------------
    x_m = np.mean(x_raw)
    y_m = np.mean(y_raw)

    t_0 = t_raw.min()

    x = x_raw - x_m
    y = y_raw - y_m
    t = t_raw - t_0

    # -----------------------------------------------------
    # 初始频率估计
    # -----------------------------------------------------
    y_c = y - np.mean(y)

    freqs = np.linspace(1, 300, 200)

    power = [
        np.abs(
            np.sum(
                y_c *
                np.exp(-1j * 2 * np.pi * f * t)
            )
        )
        for f in freqs
    ]

    f_init = freqs[np.argmax(power)]

    # -----------------------------------------------------
    # 参数初始化
    # -----------------------------------------------------
    p0 = [
        0.0,
        1.0,
        np.ptp(y) / 2,
        f_init,
        0.0,
        0.1,
        0.0
    ]

    # -----------------------------------------------------
    # 优化目标
    # -----------------------------------------------------
    def objective(p):

        A, B, C, f, phi, lamb, D = p

        vibration = (
            C *
            np.exp(-lamb * t) *
            np.sin(
                2 * np.pi * f * t + phi
            )
        )

        residual = (
            A * x +
            B * y +
            vibration +
            D
        )

        penalty = (
            1e6 *
            (np.sqrt(A**2 + B**2) - 1)**2
        )

        return np.mean(residual**2) + penalty

    # -----------------------------------------------------
    # 优化
    # -----------------------------------------------------
    res = minimize(
        objective,
        p0,
        method='L-BFGS-B',
        bounds=[
            (-1, 1),
            (-1, 1),
            (0, None),
            (1, 500),
            (-np.pi, np.pi),
            (0, 10),
            (None, None)
        ]
    )

    if not res.success:
        return None

    popt = res.x

    # -----------------------------------------------------
    # 归一化
    # -----------------------------------------------------
    norm = np.sqrt(
        popt[0]**2 +
        popt[1]**2
    )

    popt[[0,1,2,6]] /= norm

    return {
        'params': popt,
        'x_m': x_m,
        'y_m': y_m,
        't_0': t_0,
        'id': group['id'],
        'x_range': group['x_range']
    }


# =========================================================
# 5. 构造3DFIT位移信号
# =========================================================
def build_3dfit_signal(fitted_group,
                       t_start,
                       t_end,
                       dt=0.001):

    p = fitted_group['params']

    A, B, C, f, phi, lamb, D = p

    times = np.arange(
        t_start,
        t_end,
        dt
    )

    t_local = times - fitted_group['t_0']

    signal = (
        C *
        np.exp(-lamb * t_local) *
        np.sin(
            2 * np.pi * f * t_local + phi
        )
    )

    return times, signal, f, C


# =========================================================
# 6. 原始事件散点
# =========================================================
def plot_raw_events_only(roi_data):

    fig = plt.figure(figsize=(10,7))

    ax = fig.add_subplot(
        111,
        projection='3d'
    )

    ax.scatter(
        roi_data['x'],
        roi_data['t'],
        roi_data['y'],
        s=1,
        alpha=0.3,
        c='dodgerblue',
        edgecolors='none'
    )

    ax.set_xlabel("X (Pixels)")
    ax.set_ylabel("Time (s)")
    ax.set_zlabel("Y (Pixels)")

    ax.set_title(
        "Raw Event Spatiotemporal Cloud"
    )

    ax.view_init(
        elev=20,
        azim=-35
    )

    plt.tight_layout()
    plt.show()


# =========================================================
# 7. 拟合曲面
# =========================================================
def plot_fitted_surface_only(
        roi_data,
        fitted_groups):

    fig = plt.figure(figsize=(10,7))

    ax = fig.add_subplot(
        111,
        projection='3d'
    )

    for g in fitted_groups:

        p = g['params']

        A, B, C, f, phi, lamb, D = p

        x0, x1 = g['x_range']

        xg = np.linspace(
            x0,
            x1,
            60
        )

        tg = np.linspace(
            roi_data['t'].min(),
            roi_data['t'].max(),
            120
        )

        XG, TG = np.meshgrid(
            xg,
            tg
        )

        xl = XG - g['x_m']
        tl = TG - g['t_0']

        vibration = (
            C *
            np.exp(-lamb * tl) *
            np.sin(
                2 * np.pi * f * tl + phi
            )
        )

        yl = -(
            A * xl +
            vibration +
            D
        ) / B

        YG = yl + g['y_m']

        ax.plot_surface(
            XG,
            TG,
            YG,
            cmap='viridis',
            alpha=0.9,
            antialiased=True,
            rcount=100,
            ccount=100
        )

    ax.set_xlabel("X (Pixels)")
    ax.set_ylabel("Time (s)")
    ax.set_zlabel("Y (Pixels)")

    ax.set_title(
        "Fitted Implicit Damped Surface"
    )

    ax.view_init(
        elev=20,
        azim=-35
    )

    plt.tight_layout()
    plt.show()


# =========================================================
# 8. 时间位移图
# =========================================================
def plot_time_signal(times,
                     signal):

    plt.figure(figsize=(6,5))

    plt.plot(
        times,
        signal,
        lw=1.5
    )

    plt.xlabel(
        "Time (s)",
        fontsize=16
    )

    plt.ylabel(
        "Displacement (px)",
        fontsize=16
    )

    plt.title(
        "3DFIT Vibration Signal",
        fontsize=18
    )

    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.show()


# =========================================================
# 9. 3DFIT 参数频谱（非FFT）
# =========================================================
def plot_3dfit_spectrum(f_est):

    # -----------------------------------------------------
    # 构造连续频谱
    # -----------------------------------------------------
    freqs = np.linspace(
        0,
        250,
        3000
    )

    sigma = 1.5

    spectrum = np.exp(
        -(
            (freqs - f_est)**2
        ) / (2 * sigma**2)
    )

    spectrum /= (
        spectrum.max() + 1e-12
    )

    # -----------------------------------------------------
    # 可视化参数
    # -----------------------------------------------------
    grid_alpha = 0.3
    label_fontsize = 20
    tick_fontsize = 18

    # -----------------------------------------------------
    # 频谱图
    # -----------------------------------------------------
    plt.figure(figsize=(6,5))

    plt.plot(
        freqs,
        spectrum,
        lw=2
    )

    plt.scatter(
        f_est,
        1.0,
        color='r',
        s=60,
        zorder=3,
        label=f"{f_est:.2f} Hz"
    )

    plt.xlabel(
        "Frequency (Hz)",
        fontsize=label_fontsize
    )

    plt.ylabel(
        "Normalized Amplitude",
        fontsize=label_fontsize
    )

    plt.xlim(0,250)

    plt.legend(
        frameon=False,
        fontsize=tick_fontsize,
        loc='upper right'
    )

    plt.grid(alpha=grid_alpha)

    plt.tick_params(
        axis='both',
        labelsize=tick_fontsize
    )

    plt.tight_layout()

    plt.show()


# =========================================================
# 10. Main
# =========================================================
def main():

    # =====================================================
    # 数据
    # =====================================================
    # file_path = "./data/30hz-20.hdf5"
    # data = load_events(file_path)
    # roi = select_large_roi(
    #     data,
    #     100, 600,
    #     420, 460,
    #     0, 1.0
    # )

    file_path = "./data/zdbiaochi.hdf5" # 替换为你的路径
    # data = load_events(file_path)
    # roi = select_large_roi(data, 300, 1000, 400, 600, 0, 0.2)
    data = load_events(file_path)
    roi = select_large_roi(
        data,
        600, 1000,
        400, 600,
        0, 1.0
    )

    

    # file_path = './data/200hz-20.hdf5'  # 替换为你的HDF5文件路径
    # data = load_events(file_path)
    # roi = select_large_roi(
    #     data,
    #     200, 600,
    #     350, 460,
    #     0, 0.5
    # )
    
    print(
        f"Events in ROI: "
        f"{len(roi['x'])}"
    )

    # =====================================================
    # 分组
    # =====================================================
    groups = group_by_x(
        roi,
        group_size=800
    )

    fitted = []

    # =====================================================
    # 拟合
    # =====================================================
    for g in groups:

        res = fit_group_implicit(g)

        if res:

            fitted.append(res)

            p = res['params']

            print(
                f"Group {res['id']} "
                f"Fitted: "
                f"f={p[3]:.2f}Hz, "
                f"λ={p[5]:.2f}"
            )

    if not fitted:

        print("拟合失败")
        return

    # =====================================================
    # 图1：原始事件
    # =====================================================
    print("\nGenerating raw event cloud...")
    plot_raw_events_only(roi)

    # =====================================================
    # 图2：拟合曲面
    # =====================================================
    print("\nGenerating fitted surface...")
    plot_fitted_surface_only(
        roi,
        fitted
    )

    # =====================================================
    # 构造3DFIT位移信号
    # =====================================================
    times, signal, f_est, amp = \
        build_3dfit_signal(
            fitted[0],
            roi['t'].min(),
            roi['t'].max(),
            dt=0.001
        )

    # =====================================================
    # 图3：时间位移
    # =====================================================
    print("\nGenerating 3DFIT signal...")
    plot_time_signal(
        times,
        signal
    )

    # =====================================================
    # 图4：3DFIT参数频谱
    # =====================================================
    print("\nGenerating 3DFIT fitted spectrum...")
    plot_3dfit_spectrum(
        f_est
    )

    # =====================================================
    # 输出
    # =====================================================
    print("\n========== 3DFIT Result ==========")

    print(
        f"Dominant Frequency : "
        f"{f_est:.3f} Hz"
    )

    print(
        f"Estimated Amplitude : "
        f"{amp:.3f} px"
    )


# =========================================================
# main
# =========================================================
if __name__ == "__main__":

    main()